# **1. API Configuration**

In [29]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)

OpenAI client configured successfully.
Model: gpt-4.1-mini


# **2.1 Define 5 Funtions or Tools in Python Format with Return**
**Note:** Capture & Raise Expections if you think it will raise later.

In [2]:
def employee_lookup(employee_id):
    employees = {101: {"name": "Madhavan", "department": "AI Research Team"},
                 102: {"name": "Rajesh", "department": "Operations"},
                 103: {"name": "Jagadeeshwari", "department": "AI Engineering Team"}}
    return employees.get(employee_id,{"error": "Employee Not Found"})

def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves - leaves_taken}

def calculate(expression):
    allowed = set("0123456789+-*/(). %")

    if not expression or any(char not in allowed for char in expression):
        return {"error": "Only basic arithmetic expressions are allowed."}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception:
        return {"error": "Invalid arithmetic expression."}

def send_email(to, subject, body):
    return {"status": "simulated","message": f"Email prepared for {to} with subject '{subject}'." }

def book_meeting(title, attendee, time):
    return {"status": "simulated","message": f"Meeting '{title}' prepared with {attendee} at {time}."}

# **2.2 Now, define in OpenAI SDK Format**

In [3]:
TOOLS = [
    {
        "type": "function",
        "name": "employee_lookup",
        "description": "Look up an employee's name and department using an employee ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "employee_id": {"type": "integer"}
            },
            "required": ["employee_id"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate remaining leave from total and taken leaves.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {"type": "integer"},
                "leaves_taken": {"type": "integer"}
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "Perform a basic arithmetic calculation.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "send_email",
        "description": "Prepare a simulated email.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "book_meeting",
        "description": "Prepare a simulated meeting booking.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "attendee": {"type": "string"},
                "time": {"type": "string"}
            },
            "required": ["title", "attendee", "time"],
            "additionalProperties": False
        }
    }
]


Available tools: ['employee_lookup', 'calculate_leave_balance', 'calculate', 'send_email', 'book_meeting']


# **3. Send the API Request to get the Tool Call**

In [36]:
question = "Who is employee 102?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                   tools=TOOLS)

print(response.output)

for item in response.output:
    if item.type == "function_call":
        print("Selected tool:", item.name)
        print("Arguments:", item.arguments)

[ResponseFunctionToolCall(arguments='{"employee_id":102}', call_id='call_56w7BIvq11h6R6ed9s86MnVi', name='employee_lookup', type='function_call', id='fc_0dc738231d078025006a9027ce628087d0b83a6cc39360b579', caller=None, namespace=None, status='completed')]
Selected tool: employee_lookup
Arguments: {"employee_id":102}


# **What if your input now is...**

Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and department and asks HR to confirm the employee's leave balance.

# ===================================================================================

# **Prerequisite Step: When dealing with Multiple Tools**

## **Store the Tool Registery**

In [37]:
TOOL_REGISTRY = {"employee_lookup": employee_lookup,
                 "calculate_leave_balance": calculate_leave_balance,
                 "calculate": calculate,
                 "send_email": send_email,
                 "book_meeting": book_meeting}

# **INTERVIEW QUESTION:**

## **How to Create an AI Agent without using any Frameworks? Possible?**



### **10 Steps to Create and Run the Agent....**

1. Receive User Input
2. Store Conversation
3. Start Agent Loop
4. Send Request to LLM
5. Check for Tool Call
6. Save LLM Response
7. Execute Selected Tool
8. Get Tool Result
9. Send Tool Result Back to LLM
10. Repeat Until Final Answer or Maximum Steps Reached

# ***Solution Script***

In [48]:
import json

def run_agent(user_input, max_steps=5):
    input_items = [{"role": "user", "content": user_input}]

    for step in range(1, max_steps + 1):
        response = client.responses.create(model=GENERATION_MODEL,
                                           input=input_items,
                                           tools=TOOLS )

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            return response.output_text

        input_items.extend([item.model_dump() for item in response.output]) #--> part 1 work is done

        for tool_call in tool_calls:
            if tool_call.name not in TOOL_REGISTRY:
                raise ValueError(f"Unknown tool requested: {tool_call.name}")

            arguments = json.loads(tool_call.arguments)

            result = TOOL_REGISTRY[tool_call.name](**arguments)

            print(f"Step {step}")
            print("Tool:", tool_call.name)
            print("Arguments:", arguments)
            print("Result:", result)

            input_items.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(result)
            })

    return "Maximum agent steps reached."

In [50]:
run_agent("Find employee 102 and tell me their name and department.")

Step 1
Tool: employee_lookup
Arguments: {'employee_id': 102}
Result: {'name': 'Rajesh', 'department': 'Operations'}


'Employee 102 is named Rajesh, and he works in the Operations department.'

In [51]:
run_agent("Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and department and asks HR to confirm the employee's leave balance.")

Step 1
Tool: employee_lookup
Arguments: {'employee_id': 103}
Result: {'name': 'Jagadeeshwari', 'department': 'AI Engineering Team'}
Step 2
Tool: send_email
Arguments: {'to': 'hr@company.com', 'subject': 'Leave Balance Confirmation for Jagadeeshwari', 'body': 'Dear HR Team,\n\nI hope this message finds you well. Could you please confirm the leave balance for Jagadeeshwari from the AI Engineering Team? We need to verify her current leave status.\n\nThank you for your assistance.\n\nBest regards,\n[Your Name]'}
Result: {'status': 'simulated', 'message': "Email prepared for hr@company.com with subject 'Leave Balance Confirmation for Jagadeeshwari'."}


"The employee with ID 103 is Jagadeeshwari from the AI Engineering Team. I have prepared a simulated email to HR asking them to confirm Jagadeeshwari's leave balance. Would you like me to assist with anything else?"

# **THE END.**